# COMP8410 A1 - Week 7: video editing - Wan2.1-VACE-1.3B vs Lucy-Edit-Dev

Matched comparison: same clips, same instructions, same seed, **480x832, 50 steps, CFG 5.0 for both**.
The one declared asymmetry is conditioning - VACE consumes a spatial edit mask, Lucy-Edit is
instruction-only. The mask box is used at *scoring* time for both models, so `bg_psnr`
(fidelity outside the edit region) is measured identically.

**Order:** 1.1 -> 2.1 -> 2.2 -> 2.3 -> 2.4 (look at the inputs!) -> 3.x smokes -> 3.3 gate -> 4.x full -> 5.x cost.
Never use Run all. Disconnect the GPU when idle.

## 1. Setup

In [ ]:
# [1.1] SETUP, fresh runtime only: mount Drive, clone the repo, link results/ and outputs/ to Drive
%cd /content
import os
os.environ['HF_HOME'] = '/content/hf'
from google.colab import drive # type: ignore
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/comp8410_a1'
for d in ['results', 'outputs', 'logs']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
!rm -rf /content/a1 && git clone https://github.com/yash1998-MQ-AI/COMP8410-A1-48576158.git /content/a1
%cd "/content/a1/48576158 Evaluation"
!rm -rf results outputs && ln -s {BASE}/results results && ln -s {BASE}/outputs outputs
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# [1.2] RESTORE after a kernel restart (use INSTEAD of 1.1; NOT after switching runtime type)
import os
os.environ['HF_HOME'] = '/content/hf'
BASE = '/content/drive/MyDrive/comp8410_a1'
%cd "/content/a1/48576158 Evaluation"
from google.colab import drive
drive.mount('/content/drive')
!git -C /content/a1 pull -q
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install, tests, inputs

Both models are diffusers-native (no repo to clone, no flash-attn), so 2.1 is quick.

In [ ]:
# [2.1] INSTALL (diffusers >= 0.42.1 ships WanVACEPipeline and LucyEditPipeline)
%cd "/content/a1/48576158 Evaluation"
!python pairs/w7_video_editing/install_w7.py 2>&1 | tail -20

In [ ]:
# [2.2] CPU TESTS: the pure functions (frame counts, mask, PSNR/SSIM outside the box, flicker)
%cd "/content/a1/48576158 Evaluation"
!python -m pytest tests/test_w7.py -q 2>&1 | tail -5

In [ ]:
# [2.3] PREPARE INPUTS -> 832x480 mp4s at a 4n+1 frame count + items.csv
# SRC = the folder holding your walkthrough clips. Options:
#   your uploaded clips:  f'{BASE}/w7_data/setD'
#   Pair 5 clips instead: 'outputs/w6/hunyuanvideo15'
%cd "/content/a1/48576158 Evaluation"
SRC = f'{BASE}/w7_data/setD'
!ls -la "{SRC}" 2>/dev/null | head -12 || echo "SRC NOT FOUND - upload the clips or point SRC at outputs/w6/hunyuanvideo15"
!python pairs/w7_video_editing/prepare_inputs.py --src "{SRC}" --limit 6 --force 2>&1 | grep -v "pix_fmt" | tail -12
print('\n--- items.csv ---')
!cat pairs/w7_video_editing/items.csv

In [ ]:
# [2.4] LOOK AT THE INPUTS BEFORE SPENDING GPU (W6 lesson: wrong inputs cost a whole run).
# Contact sheet: first frame of each prepared clip with its edit box drawn on.
%cd "/content/a1/48576158 Evaluation"
import sys, numpy as np
from PIL import Image, ImageDraw
sys.path.insert(0, 'pairs/w7_video_editing'); sys.path.insert(0, '.')
import w7_common as C
from prepare_inputs import read_frames
ids = C.item_ids()
tiles = []
for it in ids:
    fr, fps = read_frames(C.clip_path(it))
    im = Image.fromarray(fr[0]).copy()
    d = ImageDraw.Draw(im)
    d.rectangle(C.box_pixels(C.box_of(it)), outline=(255, 40, 40), width=5)
    d.text((8, 8), f"{it}  {len(fr)}f @{fps:.0f}fps", fill=(255, 255, 0))
    tiles.append(np.asarray(im.resize((416, 240))))
    print(f"{it:16s} {len(fr):3d}f  {fr[0].shape[1]}x{fr[0].shape[0]}  box={C.box_of(it)}  '{C.prompt_of(it)[:58]}'")
rows = [np.hstack(tiles[i:i+3]) for i in range(0, len(tiles), 3)]
w = max(r.shape[1] for r in rows)
rows = [np.pad(r, ((0,0),(0,w-r.shape[1]),(0,0))) for r in rows]
display(Image.fromarray(np.vstack(rows)))

## 3. Smoke tests and the budget gate (A100 or L4)

6 steps instead of 50 and one item, so each smoke is a couple of minutes. Check the output
clip actually changed inside the box and not outside before committing to the full run.

In [ ]:
# [3.1] SMOKE wan21_vace (mask-conditioned): 1 item, 6 steps. First run downloads the model.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w7_video_editing/gen_w7.py wan21_vace --smoke 2>&1 | grep -v Warning | tail -12

In [ ]:
# [3.2] SMOKE lucy_edit (instruction-only): 1 item, 6 steps.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w7_video_editing/gen_w7.py lucy_edit --smoke 2>&1 | grep -v Warning | tail -12

In [ ]:
# [3.3] LOOK AT THE SMOKE OUTPUTS + BUDGET GATE.
# Top row = input first frame, then each model's first and last frame. Then the projection.
%cd "/content/a1/48576158 Evaluation"
import sys, json, glob
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
sys.path.insert(0, 'pairs/w7_video_editing'); sys.path.insert(0, '.')
import w7_common as C
from prepare_inputs import read_frames

item = C.item_ids()[0]
panels, labels = [], []
fr, _ = read_frames(C.clip_path(item))
panels += [fr[0]]; labels += [f'INPUT {item}']
lat = {}
for m in C.MODELS:
    p = Path('outputs/w7_smoke') / m / f'{item}_s0.mp4'
    side = p.with_suffix('.json')
    if side.exists():
        d = json.loads(side.read_text())
        lat[m] = d.get('latency_s')
        print(f"{m:12s} ok={d.get('ok')} latency={d.get('latency_s', 0):.0f}s vram={d.get('peak_vram_gb', 0):.1f}GB "
              f"steps={d.get('steps')} {d.get('error', '')}")
    if p.exists():
        g, _ = read_frames(p)
        panels += [g[0], g[-1]]; labels += [f'{m} f0', f'{m} last']
    else:
        print(f"{m}: NO OUTPUT at {p}")
if len(panels) > 1:
    tiles = []
    for a, lb in zip(panels, labels):
        im = Image.fromarray(a).resize((416, 240))
        ImageDraw.Draw(im).text((8, 8), lb, fill=(255, 255, 0))
        tiles.append(np.asarray(im))
    rows = [np.hstack(tiles[i:i+3]) for i in range(0, len(tiles), 3)]
    w = max(r.shape[1] for r in rows)
    display(Image.fromarray(np.vstack([np.pad(r, ((0,0),(0,w-r.shape[1]),(0,0))) for r in rows])))

# Projection: smoke used 6 steps, the full run uses C.STEPS. Sampling time scales with steps.
UNITS_PER_HR = 11.77   # A100-40GB. Change to your Resources-panel rate (T4 ~1.1, L4 ~4.8).
BALANCE = None         # <- put your remaining compute units here to get a verdict
n_items, n_seeds = len(C.item_ids()), len(C.SEEDS)
if lat:
    per_item = sum(lat.values()) * (C.STEPS / 6)          # both models, scaled to full steps
    total_s = per_item * n_items * n_seeds + 2 * 180      # + model load allowance
    units = total_s / 3600 * UNITS_PER_HR
    print(f"\nprojected full run: {total_s/60:.0f} min for {n_items} items x {n_seeds} seed(s) x 2 models")
    print(f"projected cost: {units:.1f} units = A${units*0.1513:.2f}")
    if BALANCE is not None:
        print('GATE PASSED' if units < BALANCE * 0.8 else f'GATE FAILED - reduce --limit or steps (balance {BALANCE})')
    else:
        print('set BALANCE to your remaining units for a verdict')

## 4. Full runs, scoring, report

In [ ]:
# [4.1] FULL wan21_vace: all items, 50 steps. Resumable (a sidecar json marks each done clip).
%cd "/content/a1/48576158 Evaluation"
!python pairs/w7_video_editing/gen_w7.py wan21_vace 2>&1 | grep -v Warning | tee {BASE}/logs/w7_gen_vace.log | tail -12

In [ ]:
# [4.2] FULL lucy_edit: all items, 50 steps. Resumable.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w7_video_editing/gen_w7.py lucy_edit 2>&1 | grep -v Warning | tee {BASE}/logs/w7_gen_lucy.log | tail -12

In [ ]:
# [4.3] SCORE every edited clip (CLIP + DINOv2 + pixel metrics). Safe to re-run; --rescore forces.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w7_video_editing/score_w7.py 2>&1 | grep -v Warning | tail -20

In [ ]:
# [4.4] TABLES AND PLOTS (copies report assets and pip freeze to Drive)
%cd "/content/a1/48576158 Evaluation"
!python pairs/w7_video_editing/report_w7.py 2>&1 | tee {BASE}/logs/w7_report.log | grep -v Warning | tail -40
!mkdir -p {BASE}/report_assets && cp -r report_assets/w7 {BASE}/report_assets/
!pip freeze > {BASE}/results/w7_ENV.txt

In [ ]:
# [4.5] QUALITATIVE STRIPS for the report: input vs both models, 4 frames each, per item.
%cd "/content/a1/48576158 Evaluation"
import sys
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
sys.path.insert(0, 'pairs/w7_video_editing'); sys.path.insert(0, '.')
import w7_common as C
from prepare_inputs import read_frames
out = Path('outputs/w7_figures'); out.mkdir(parents=True, exist_ok=True)
for it in C.item_ids():
    rows = []
    for label, path in [('input', C.clip_path(it))] + [(m, C.video_path(m, it, C.SEEDS[0])) for m in C.MODELS]:
        if not Path(path).exists():
            continue
        fr, _ = read_frames(Path(path))
        idx = [0, len(fr)//3, 2*len(fr)//3, len(fr)-1]
        tiles = []
        for j, i in enumerate(idx):
            im = Image.fromarray(fr[i]).resize((320, 185))
            if j == 0:
                ImageDraw.Draw(im).text((6, 6), label, fill=(255, 255, 0))
            tiles.append(np.asarray(im))
        rows.append(np.hstack(tiles))
    if rows:
        sheet = Image.fromarray(np.vstack(rows))
        sheet.save(out / f'w7_strip_{it}.jpg', quality=90)
        print(f'{it}: {sheet.size}')
        display(sheet)

## 5. Cost, shut down

In [ ]:
# [5.1] COST LOG: fill in HOURS and UNITS_PER_HR (Resources panel), run once per session
%cd "/content/a1/48576158 Evaluation"
HOURS = None
UNITS_PER_HR = None
if HOURS and UNITS_PER_HR:
    from a1eval import log_cost
    log_cost('cost_log.csv', '2026-11-05', 'Colab Pro', 'A100-40GB', HOURS, UNITS_PER_HR * 0.1513,
             note='W7 video editing: Wan2.1-VACE-1.3B vs Lucy-Edit-Dev')
    !tail -4 cost_log.csv

In [ ]:
# [5.2] FREE THE GPU: set AUTO_DISCONNECT = True once everything is copied to Drive
AUTO_DISCONNECT = False
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()